In [3]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 58.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 66.6 MB/s eta 0:00:0000:0100:01


In [4]:
import json
import pandas as pd

In [6]:
df = pd.read_json('/home/cmpdil/Iit_profbehra/TinyGPT-ECG/TinyGPT-V-main/data/stage3.json')
print(df.head())

                                            question  \
0  Locate the Premature ventricular contraction o...   
1  Examine this ECG and point out where the Prema...   
2  Where should I look to find the Premature vent...   
3  Assess this ECG and specify the location of th...   
4  Can you locate the Premature ventricular contr...   

                            answer ecg_path  \
0   Duration: 0.4s-3.7s, 4.6s-9.3s  [e0103]   
1  Duration: 1.8s-5.1s, 6.1s-10.0s  [e0103]   
2   Duration: 0.0s-3.1s, 4.1s-8.8s  [e0103]   
3   Duration: 0.3s-3.6s, 4.6s-9.3s  [e0103]   
4   Duration: 0.0s-3.3s, 4.3s-9.0s  [e0103]   

                                             dataset      task abnormal_type  \
0  /home/cmpdil/iit_profbehra2/ecg_data/european-...  location             V   
1  /home/cmpdil/iit_profbehra2/ecg_data/european-...  location             V   
2  /home/cmpdil/iit_profbehra2/ecg_data/european-...  location             V   
3  /home/cmpdil/iit_profbehra2/ecg_data/european-...  locati

In [7]:
df["dataset"] = df["dataset"].replace("/home/cmpdil/iit_profbehra2/ecg_data/physionet.org/","/home/cmpdil/iit_profbehra2/ecg_data/physionet.org/files/ptb-xl/1.0.3/")


In [8]:
df.to_json('stage3.json', orient='records', indent=4)

In [1]:
import pandas as pd
import ast

# Load raw dataset and statement mapping from PhysioNet
df = pd.read_csv('/home/cmpdil/iit_profbehra2/ecg_data/physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv')
mapping = pd.read_csv('/home/cmpdil/iit_profbehra2/ecg_data/physionet.org/files/ptb-xl/1.0.3/scp_statements.csv', index_col=0)

# Convert string representation of dictionaries to Python dictionaries
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)

# Function to map scp codes to their full string descriptions
def get_diagnostic_class(scp_dict):
    res = []
    for code in scp_dict.keys():
        if code in mapping.index:
            res.append(mapping.loc[code, 'description'])
    return res

# Apply mapping
df['diagnoses'] = df['scp_codes'].apply(get_diagnostic_class)

# Explode the list of diagnoses into individual one-hot encoded columns
mlb = pd.Series(df['diagnoses']).explode()
crosstab = pd.crosstab(mlb.index, mlb)
df = df.join(crosstab)

# Save the processed dataset for the pipeline
df.to_csv('ptbxl_processed.csv', index=False)

In [1]:
import pandas as pd

# Load the previously processed dataset
df = pd.read_csv('ptbxl_processed.csv')

# Map the numeric folds to string partitions
def map_splits(fold_num):
    if fold_num == 10:
        return 'test'
    elif fold_num == 9:
        return 'val'
    else:
        return 'train'

# Apply the mapping logic
df['split_fold'] = df['split_fold'].apply(map_splits)

# Save the modifications
df.to_csv('ptbxl_processed.csv', index=False)